<h1 align="center">Cell Types and Learning: linking transcriptomic identity to neural activity</h1>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h2>Overview</h2>

In the Cell Types and Learning (CTL) dataset, the same neurons are measured twice:

1. **In vivo**, with two-photon calcium imaging, while a mouse performs a visual change-detection task.
2. **Post hoc**, with spatial transcriptomics (HCR), which tells us which *cell type* each neuron is.

A coregistration pipeline links the two. That means we can ask a question you cannot ask
with either measurement alone: **does a neuron's transcriptomic cell type predict what it
does during behavior?**

This notebook is about the two kinds of alignment that question requires:

- **Across modalities** — matching an imaged neuron to its transcriptomic cell type (Part 3)
- **Across sessions** — matching a neuron imaged on one day to the same neuron on another day (Part 7)

Once the alignment is done, we make the same four plots for each session type:

| Plot | What it shows |
| --- | --- |
| Max projections by depth | where the neurons are, coloured by cell type |
| dF/F heatmaps | all activity in the session, then sorted by cell type |
| Tuning heatmap | which stimulus each neuron prefers |
| Change-aligned response | activity locked to the moment the image changes |

We do this first for a **gratings** session, then for two **natural image** sessions, and finally
compare the same neurons across two of those sessions.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h2>The data</h2>

Code Ocean mounts each attached data asset read-only under `/data`, in a folder named
after the asset. This tutorial needs three, plus the metadata tables:

| Asset | What it provides |
| --- | --- |
| `multiplane-ophys-behavior-nwb-combined-asset-swdb` | one NWB per imaging session: activity, behavior, ROI masks |
| `cell-types-and-learning_coreg-id-mapping-assets-combined_2025-07-06` | the ID table linking imaged ROIs to HCR cells |
| `HCR_<mouse>_unmixed-calibrated_*` | the cell x gene **AnnData**, carrying the cell-type labels |

The division of labour between the last two is worth being explicit about, because it is the
part people get wrong:

- The **coregistration asset** answers *"which HCR cell is this ROI?"*. It is a table of
  **identifiers only** — no expression, no cell types.
- The **HCR AnnData** answers *"what type is that HCR cell, and what genes does it express?"*.
  It knows nothing about imaging.

Neither is useful alone. The coreg table's `hcr_id` is the key that opens the AnnData.

If a folder is missing, the asset is not attached — attach it in the capsule's Data panel.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h2>Setup</h2>

</div>

In [1]:
import os
import glob
import json

import numpy as np
import pandas as pd
import anndata as ad
import pynwb
import matplotlib.pyplot as plt

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)

data_dir = '/data'

# the 8 mesoscope imaging planes
planes = [f'VISp_{i}' for i in range(8)]

# the inhibitory subclasses the HCR gene panel resolves, in standard order.
# These are the labels used in the AnnData `subclass` column.
subclass_order = ['Pvalb', 'Sst', 'Vip', 'Lamp5']
subclass_colors = {'Pvalb': '#D93137', 'Sst': '#FF9900',
                   'Vip': '#A45FBF', 'Lamp5': '#DA808C'}

# the peri-change window used for every alignment in this notebook
window = np.arange(-2, 4, 0.1)

plt.rcParams.update({
    'font.size': 14, 'axes.titlesize': 16, 'axes.labelsize': 15,
    'xtick.labelsize': 13, 'ytick.labelsize': 13, 'legend.fontsize': 13,
    'figure.titlesize': 18, 'figure.dpi': 100,
})

# Each dataset attached to this capsule appears as its own directory under /data.
sorted(os.listdir(data_dir))

['409828_V1DD_Filtered',
 '416296_V1DD_Filtered',
 '427836_V1DD_Filtered',
 '438833_V1DD_Filtered',
 'Neuropixels_Opto_ecephys_nwb_combined',
 'Visual-Learning-SWDB',
 'brain-computer-interface-v2',
 'cell_types_and_learning_coreg_cellxgene_table_all_mice.csv',
 'dynamicrouting_datacube',
 'metadata']

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h2>Part 1: Find sessions using the metadata table</h2>

Before opening any data, look at the **session metadata table**. It has one row per session
across the whole dataset, with the session type, the stimulus, how many neurons were found,
and so on. It is the fastest way to find sessions worth analyzing.

There are two metadata tables and we use both:

- `ctl_session_metadata_*.csv` — **one row per session** (session type, stimulus, trial counts)
- `ctl_session_plane_metadata_*.csv` — **one row per session x imaging plane** (imaging depth,
  frame rate, per-plane ROI counts and QC)

</div>

In [3]:
sessions = pd.read_csv(os.path.join(data_dir, 'metadata', 'visual_learning_session_metadata.csv'))

print('sessions      ', sessions.shape)
print('mice          ', sorted(sessions['subject_id'].unique()))

sessions       (147, 18)
mice           [np.int64(782149), np.int64(788406), np.int64(790322), np.int64(800792), np.int64(800995), np.int64(804363)]


<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

The session-table columns we care about here:

| Column | Meaning |
| --- | --- |
| `subject_id` | which mouse |
| `session_number` | order the sessions were acquired in, per mouse |
| `session_type` | the training or imaging stage |
| `stimulus_category` | what was on the screen |
| `image_set` | which set of natural images (A or B), if any |
| `session_id` | the acquisition name, which is how the NWB file is named |

Pick one mouse and list its sessions in acquisition order.

</div>

In [5]:
mouse = 800995

mouse_sessions = sessions[sessions['subject_id'] == mouse].sort_values('session_number')

mouse_sessions[['session_number', 'session_date', 'session_type']]

,session_number,session_date,session_type
105,1,2025-08-05,TRAINING_0_gratings_autorewards_15min
106,2,2025-08-07,TRAINING_1_gratings
107,3,2025-08-08,TRAINING_1_gratings
108,4,2025-08-11,TRAINING_1_gratings
109,5,2025-08-12,TRAINING_2_gratings_flashed
110,6,2025-08-13,TRAINING_3_images_A_10uL_reward
111,7,2025-08-14,TRAINING_3_images_A_10uL_reward
112,8,2025-08-15,TRAINING_3_images_A_10uL_reward
113,9,2025-08-18,TRAINING_4_images_A_training
114,10,2025-08-19,TRAINING_5_images_A_epilogue


<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

This mouse's history reads as a training curriculum:

1. **`TRAINING_*_gratings`** — learning the task with simple grating stimuli
2. **`TRAINING_*_images_A`** — the same task with natural images
3. **`OPHYS_1_images_A`** — task performed with the now-**familiar** image set A
4. **`OPHYS_4_images_B`** — the same task with a **novel** image set B
5. **`OPHYS_6_images_B`** — set B again, once it too has become familiar

Rather than hardcoding dates, **select the sessions by querying the table** — the same code then
works for any mouse. We take the first session of each type, since for the novel session
"first" is what makes it novel.

</div>

In [7]:
def pick_session(session_type):
    """First session of a given type for this mouse, as a metadata row."""
    matches = mouse_sessions[mouse_sessions['session_type'] == session_type]
    if matches.empty:
        raise ValueError(f'mouse {mouse} has no {session_type} session')
    return matches.iloc[0]


# TRAINING_1: gratings have an orientation, so we can measure orientation tuning
# OPHYS_1:    familiar natural images (set A)
# OPHYS_4:    FIRST exposure to novel images (set B)
GRATINGS = pick_session('TRAINING_1_gratings')
FAMILIAR = pick_session('OPHYS_1_images_A')
NOVEL    = pick_session('OPHYS_4_images_B')

for row in [GRATINGS, FAMILIAR, NOVEL]:
    print(f"session {row['session_number']:>2}  {row['session_date']}  "
          f"{row['session_type']:<24} "
          f"{row['n_trials']:.0f} trials")

KeyError: 'n_trials'

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h2>Part 2: Load a session</h2>

Find the NWB file for a session. The file is named after the `session_id` from the metadata
table, so we can walk the asset looking for that string.

The NWB may be a single `.nwb` file **or** a `.nwb.zarr` **directory** — Zarr stores an array as
many small chunk files in a directory tree, which is what lets us read one piece of a big dataset
without loading all of it. `pynwb.read_nwb` opens either, but a directory walk has to know not to
descend into a `.nwb.zarr`, because from the outside it looks like a folder.

</div>

In [ ]:
def find_nwb(root, contains=None, max_depth=3):
    """Return paths to .nwb files and .nwb.zarr directories under `root`.

    `contains` filters to paths containing that string -- pass the session id.
    """
    found = []
    root = root.rstrip('/')
    base_depth = root.count(os.sep)
    for dirpath, dirnames, filenames in os.walk(root):
        if dirpath.count(os.sep) - base_depth >= max_depth:
            dirnames[:] = []
        # A .nwb.zarr directory IS the file -- do not descend into its internals.
        for d in list(dirnames):
            if d.endswith('.nwb.zarr'):
                found.append(os.path.join(dirpath, d))
                dirnames.remove(d)
        for f in filenames:
            if f.endswith('.nwb'):
                found.append(os.path.join(dirpath, f))
    if contains is not None:
        found = [p for p in found if contains in p]
    return sorted(found)


# EDIT: the mount holding the ophys NWBs -- one of the names listed above.
nwb_asset = os.path.join(data_dir, 'Visual-Learning-SWDB')

matches = find_nwb(nwb_asset, contains=GRATINGS['session_id'])
print(f'{len(matches)} NWB file(s) for {GRATINGS["session_id"]}')
for p in matches:
    print(' ', os.path.relpath(p, data_dir))

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

Open it and look at what is inside **before** indexing into it. The containers we will use:

- `processing` — the processed neural activity, one group per imaging plane
- `intervals` — tables of trials and stimulus presentations

This session was imaged with a **mesoscope**, which records 8 planes at different depths at the
same time. Each plane is a separate group under `processing`.

</div>

In [ ]:
nwb = pynwb.read_nwb(matches[0])

print('processing :', list(nwb.processing.keys()))
print('intervals  :', list(nwb.intervals.keys()) if nwb.intervals else [])
print('acquisition:', list(nwb.acquisition.keys()))

In [ ]:
# each plane group holds several versions of the activity traces
list(nwb.processing['VISp_2'].data_interfaces.keys())

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

`dff_timeseries` is the one to use for most analyses.

**&Delta;F/F** is the change in fluorescence relative to each neuron's own baseline
fluorescence. Dividing by baseline makes neurons comparable to each other: a bright neuron
and a dim one both read out as *fractional* change.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>One function to load a session</h3>

We will load three sessions, so collect the loading into a function now.

The key output is `dff`, of shape **(frames, neurons)**, holding all 8 planes side by side, plus
a `neurons` table with one row per column of `dff`. Each plane has its own timestamps, because the
mesoscope visits the planes in sequence.

The important design decision is in the `neurons` table. Segmentation gives us an ROI table per
plane whose **row order is the only thing tying it to the activity array** — the `id` column in
these files is all zeros. Rather than carrying that fragile positional relationship around, we
convert position into a **name**, once, at load time, in exactly the format the coregistration
table uses:

| Column | Built from | Example |
| --- | --- | --- |
| `roi_id` | plane name + row index, zero-padded to 4 digits | `VISp_0_0010` |
| `unique_roi_id` | `<mouse>_<date>_` + `roi_id` | `800995_2025-08-21_VISp_0_0010` |

From that point on every join is **on a string key**, not on an array offset. This is what makes
it safe to subset the data — filter to somas, filter to coregistered cells, drop a bad plane — in
any order, because each row carries its own identity with it. Positional indexing has to be got
right exactly once, here, and the assert below is what checks it.

</div>

In [ ]:
def plane_depths(session_id):
    """Imaging depth in microns for each plane, from the plane metadata table."""
    rows = plane_metadata[plane_metadata['session_id'] == session_id]
    if rows.empty:
        raise ValueError(f'no plane metadata for {session_id}')
    return dict(zip(rows['plane'], rows['imaging_depth_um'].astype(int)))


def timestamps_of(series, n_frames):
    """Timestamps for a TimeSeries, reconstructed from `rate` if not stored explicitly."""
    if series.timestamps is not None:
        return np.asarray(series.timestamps[:])
    return np.arange(n_frames) / series.rate + (series.starting_time or 0.0)


def load_session(row):
    """Load one session: dF/F for all planes, timestamps, an ROI table, and trials.

    `row` is a row of the session metadata table (use pick_session).
    """
    paths = find_nwb(nwb_asset, contains=row['session_id'])
    if not paths:
        raise FileNotFoundError(f'no NWB found for {row["session_id"]}')
    nwb = pynwb.read_nwb(paths[0])

    depths = plane_depths(row['session_id'])
    session_key = f"{mouse}_{row['session_date']}"

    dff_list, label_list, timestamps = [], [], {}

    for plane in planes:
        series = nwb.processing[plane]['dff_timeseries']['dff_timeseries']
        plane_dff = np.asarray(series.data[:])
        timestamps[plane] = timestamps_of(series, plane_dff.shape[0])

        roi_table = (nwb.processing[plane]['image_segmentation']
                     .plane_segmentations['roi_table'].to_dataframe())

        # THE positional assumption, asserted once and then never relied on again:
        # row i of the ROI table is column i of the dF/F matrix.
        assert len(roi_table) == plane_dff.shape[1], (
            f'{plane}: roi_table has {len(roi_table)} rows but dff has '
            f'{plane_dff.shape[1]} columns -- cannot assign ROI ids')

        roi_index = np.arange(len(roi_table))

        dff_list.append(plane_dff)
        label_list.append(pd.DataFrame({
            'plane': plane,
            'depth': depths[plane],
            # position turned into a name, in the coreg table's format
            'roi_id': [f'{plane}_{i:04d}' for i in roi_index],
            'unique_roi_id': [f'{session_key}_{plane}_{i:04d}' for i in roi_index],
            'column': None,      # filled in below, once planes are concatenated
            'roi_index': roi_index,
            'is_soma': roi_table['is_soma'].values.astype(bool),
        }))

    rois = pd.concat(label_list, ignore_index=True)
    rois['column'] = np.arange(len(rois))     # column into the concatenated dff

    assert rois['unique_roi_id'].is_unique, 'unique_roi_id is not unique'

    return {'session_id': row['session_id'],
            'session_key': session_key,
            'date': row['session_date'],
            'session_type': row['session_type'],
            'nwb': nwb,
            'dff': np.concatenate(dff_list, axis=1),
            'timestamps': timestamps,
            'rois': rois,
            'trials': nwb.intervals['trials'].to_dataframe(),
            'stimulus': nwb.intervals['stimulus_presentations'].to_dataframe(),
            'depths': depths,
            # planes ordered from the brain surface downward, for plotting
            'planes_by_depth': sorted(planes, key=lambda p: depths[p])}

In [ ]:
gratings = load_session(GRATINGS)

print(gratings['date'], gratings['session_type'])
print('dff shape (frames, neurons):', gratings['dff'].shape)
print('soma ROIs: %d of %d' % (gratings['rois']['is_soma'].sum(), len(gratings['rois'])))

gratings['rois'].head(3)

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h2>Part 3: Align ophys to transcriptomics</h2>

This is the first of the two alignments, and the reason the dataset exists. Everything we plot
afterwards depends on getting it right.

The link is made by a **coregistration table**, produced by matching the imaged neurons to cells
in the post-hoc HCR volume.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>The four ID systems</h3>

This is the part that trips people up. The imaging and the transcriptomics are two separate
measurements of the same tissue, and connecting them takes an intermediate step.

The physical chain is:

```
imaging plane  -->  structural stack  -->  HCR volume
  (one FOV,           (one z-stack of        (thin sections,
   one session)        the same volume)       gene expression)
```

Each imaging plane is registered into a **structural stack** — a high-resolution z-stack of the
same volume — and the structural stack is registered to the **HCR volume**. Going straight from a
2-photon plane to a tissue section is not tractable; the stack is the common reference frame that
makes both registrations possible.

So there are four identifiers, one per stage:

| ID | Scope | What it identifies |
| --- | --- | --- |
| `unique_roi_id` | **one session** | a row/column position in *this* session's arrays |
| `unique_roicat_id` | **all sessions of one mouse** | a physical neuron, tracked across days |
| `cz_stack_id` | the structural stack | the cell in the structural volume |
| `hcr_id` | the mouse's HCR volume | the transcriptomic cell |

Why two IDs on the imaging side? Because a neuron imaged on Monday and again on Tuesday is the
*same cell* but a *different row* in each day's data. `unique_roi_id` is the row;
`unique_roicat_id` is the cell.

The coregistration table has all four on the same row, which is what makes the link possible:

```
unique_roi_id  -->  unique_roicat_id  -->  cz_stack_id  -->  hcr_id  -->  subclass
  (array row)         (the neuron)        (stack cell)     (HCR cell)    (cell type)
   \_________________ coreg id mapping table _________________/   \__ HCR AnnData __/
```

We need three ends of the chain: `unique_roi_id` to find the data, `unique_roicat_id` to track a
neuron across sessions (Part 7), and `hcr_id` to look up the cell type. `cz_stack_id` is worth
knowing about because it is where a coregistration failure usually happens, and `max_iou` scores
how well that step went.

</div>

In [ ]:
coreg_asset = os.path.join(
    data_dir, 'cell-types-and-learning_coreg-id-mapping-assets-combined_2025-07-06')

coreg_path = glob.glob(os.path.join(coreg_asset, '**', f'{mouse}_coreg_id_mapping_table.csv'),
                       recursive=True)[0]

coreg_all = pd.read_csv(coreg_path, index_col=0)

print(coreg_all.shape, '|', coreg_all['session_key'].nunique(), 'sessions',
      '|', coreg_all['unique_roicat_id'].nunique(), 'distinct cells')
print('matched: %d of %d rows' % (coreg_all['matched'].sum(), len(coreg_all)))

coreg_all[['session_key', 'roi_id', 'unique_roi_id', 'unique_roicat_id',
           'plane_id', 'matched', 'max_iou', 'cz_stack_id', 'hcr_id']].head(5)

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>What is in the coregistration table</h3>

One row per **ROI per session**: every ROI that segmentation found, in every session of this
mouse, whether or not it was successfully matched to anything.

| Column | Scope | Meaning |
| --- | --- | --- |
| `session_key` | — | `<mouse>_<date>`, which session this row belongs to |
| `session_name` | — | the full acquisition + processing folder name |
| `roi_id` | one session, one plane | `VISp_<plane>_<index>`, the ROI as segmented in this plane |
| `unique_roi_id` | one session | `<mouse>_<date>_<roi_id>`, globally unique across the mouse |
| `unique_roicat_id` | **all sessions** | the **cell**, matched across sessions by ROICaT |
| `plane_id` | one session | which imaging plane, `VISp_0`-`VISp_7` |
| `matched` | — | whether this ROI was matched through the chain |
| `original_cz_stack_id` | structural stack | the raw stack match, before ambiguities are resolved |
| `resolved_cz_stack_id` | structural stack | the stack match after resolution |
| `cz_stack_id` | structural stack | the stack cell to use |
| `max_iou` | — | overlap of the ROI with its structural-stack match: match quality |
| `hcr_id` | HCR volume | the transcriptomic cell — **the key into the AnnData** |
| `undecided`, `changed` | — | bookkeeping from the ambiguity-resolution step |
| `failed_session_name`, `failed_ind` | — | where a match failed, when it did |

**Unmatched entries are `-1`, not empty.** So `hcr_id == -1` means "no transcriptomic match", and
if you join without filtering it first, every failed ROI joins to whatever sits at `-1`.

<h4>The two ID scopes, and why both exist</h4>

This is the distinction to hold onto:

- **`unique_roi_id` is per session, per plane.** It names a *detection*: this blob of pixels, in
  this plane, on this day. A given session has one row per ROI.
- **`unique_roicat_id` is per cell, across all sessions.** It names a *neuron*. ROICaT matches ROIs
  across the mouse's sessions, so one `unique_roicat_id` gathers up all the sessions in which that
  neuron was detected — which may be one session, several, or all of them.

So the relationship is **many `unique_roi_id` to one `unique_roicat_id`**, and the count varies per
cell. A neuron detected on 12 of 20 days has 12 rows sharing one `unique_roicat_id`. This is why
Part 7 can track a neuron across days at all, and why the number of sessions a cell appears in is
itself a data-quality variable.

Note also that `roi_id` is not comparable between sessions: `VISp_0_0010` on Monday and
`VISp_0_0010` on Tuesday are unrelated detections. Only `unique_roicat_id` crosses sessions.

</div>

In [ ]:
# one row per ROI per session; many unique_roi_ids map to one unique_roicat_id
per_cell = (coreg_all.groupby('unique_roicat_id')['session_key'].nunique()
            .value_counts().sort_index())
print('sessions per cell (n_sessions -> n_cells):')
print(per_cell.to_string())

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>The cell types come from the HCR AnnData</h3>

The cell-type labels live in the HCR unmixed-calibrated asset, as an **AnnData** object
(`<mouse>_cellxgene_annotated.h5ad`) rather than a CSV. AnnData is the standard container for
single-cell data and it keeps three things together, which is why we prefer it:

| Part | Shape | Contents |
| --- | --- | --- |
| `A.X` | cells x genes | expression, raw spot counts per cell |
| `A.layers['normalized']` | cells x genes | the same, normalized |
| `A.obs` | cells x annotations | `class`, `subclass`, `cluster`, QC columns |
| `A.var` | genes x annotations | `round`, `channel`, `gene` |

Crucially **`A.obs_names` is the `cell_id`, and `cell_id` is the same identifier as the coreg
table's `hcr_id`.** That single fact is the whole join.

Note that `var_names` are *probe* names like `R5-514-Pvalb` — round, channel, gene — because a
gene is measured in a particular round and channel. The plain gene symbol is in `A.var['gene']`.

</div>

In [ ]:
hcr_path = glob.glob(os.path.join(data_dir, '**', f'{mouse}_cellxgene_annotated.h5ad'),
                     recursive=True)[0]

adata = ad.read_h5ad(hcr_path)

print(adata)
print('\nobs_names (= cell_id = hcr_id):', adata.obs_names[:5].tolist())

In [ ]:
# the cell-type annotations we will use
adata.obs[['class', 'subclass', 'cluster']].value_counts().head(12)

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

Two conventions to note in `obs`:

- `subclass` is `'none'` for every cell that is not one of the four inhibitory subclasses the
  panel resolves — excitatory cells and unassigned cells alike. `'none'` is a **string, not a
  missing value**, so it will not be dropped by `dropna()`. We convert it to `NaN` on the way in.
- `class` (`excitatory` / `inhibitory` / `unassigned`) is the coarser label, and it is the honest
  place to look at how many coregistered cells got a confident call at all.

</div>

In [ ]:
cell_types = adata.obs[['class', 'subclass', 'cluster']].copy()
cell_types.columns = ['cell_class', 'subclass', 'cluster_name']

# 'none' is a placeholder string, not a subclass -- make it a real missing value
cell_types['subclass'] = cell_types['subclass'].astype(str).replace('none', np.nan)
cell_types['cluster_name'] = cell_types['cluster_name'].astype(str).replace('unassigned', np.nan)

# hcr_id is an integer in the coreg table and a string index here -- match the types
cell_types.index = cell_types.index.astype(np.int64)
cell_types.index.name = 'hcr_id'

print(len(cell_types), 'HCR cells with annotations')
cell_types.head(3)

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Joining, then subsetting</h3>

Now put the two together. Because `load_session` gave every ROI a `unique_roi_id` in the coreg
table's own format, the ophys-to-transcriptomics link is a **string merge** — no array arithmetic:

```
session rois.unique_roi_id  ==  coreg.unique_roi_id      (both '800995_2025-08-21_VISp_0_0010')
                                coreg.hcr_id  ==  adata.obs_names
```

`add_cell_types` does that merge and returns the ROI table annotated. It does **not** touch the
activity arrays — annotation and subsetting are separate steps, in that order.

</div>

In [ ]:
def add_cell_types(session):
    """Annotate a session's ROI table with coregistration and cell-type columns.

    Adds one row per ROI: unique_roicat_id, hcr_id, cz_stack_id, max_iou, and the
    cell-type labels. ROIs with no coregistration get NaN. Nothing is dropped here.
    """
    table = coreg_all[coreg_all['session_key'] == session['session_key']].copy()
    if table.empty:
        raise ValueError(f"no coregistration rows for {session['session_key']}")

    # unmatched entries are -1, not empty -- make them missing before any join
    for column in ['hcr_id', 'cz_stack_id']:
        table[column] = table[column].where(table[column] > 0)

    table = table[table['matched'].astype(bool)]

    # HCR cell --> cell type, from the AnnData annotations
    table = table.merge(cell_types, left_on='hcr_id', right_index=True,
                        how='left', validate='many_to_one')

    keep = ['unique_roi_id', 'unique_roicat_id', 'cz_stack_id', 'hcr_id', 'max_iou',
            'cell_class', 'cluster_name', 'subclass']

    # merge on the ID, NOT on (plane, position)
    session['rois'] = session['rois'].drop(
        columns=[c for c in keep if c != 'unique_roi_id' and c in session['rois']]
    ).merge(table[keep], on='unique_roi_id', how='left', validate='one_to_one')

    return session


gratings = add_cell_types(gratings)

rois = gratings['rois']
print(f"{len(rois)} segmented ROIs")
print(f"  {rois['unique_roicat_id'].notna().sum():>5} in the coregistration table")
print(f"  {rois['cz_stack_id'].notna().sum():>5} matched to a structural-stack cell")
print(f"  {rois['hcr_id'].notna().sum():>5} matched to an HCR cell")
print(f"  {rois['subclass'].notna().sum():>5} with an inhibitory subclass label")

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Is `is_soma` related to coregistration?</h3>

Before subsetting, look at how the two ROI filters interact. `is_soma` is a per-ROI
classification from segmentation: some ROIs are cell bodies, others are dendrites, neuropil, or
segmentation artefacts. Coregistration is a separate process. If the two were independent we would
expect the soma fraction to be the same before and after coregistration.

It is not, and the direction is worth knowing.

</div>

In [ ]:
soma_coreg = pd.crosstab(
    rois['is_soma'],
    pd.Series(np.where(rois['hcr_id'].notna(), 'has hcr_id',
              np.where(rois['unique_roicat_id'].notna(), 'coreg, no hcr', 'not coregistered')),
              index=rois.index, name='coregistration'),
    margins=True)

print(soma_coreg.to_string())
print()

stages = {
    'all segmented ROIs': rois['unique_roi_id'].notna(),
    'in coreg table': rois['unique_roicat_id'].notna(),
    'has cz_stack_id': rois['cz_stack_id'].notna(),
    'has hcr_id': rois['hcr_id'].notna(),
}
funnel = pd.DataFrame([
    {'stage': name,
     'n': int(mask.sum()),
     'soma': int((mask & rois['is_soma']).sum()),
     'non_soma': int((mask & ~rois['is_soma']).sum()),
     'pct_soma': round(100 * rois.loc[mask, 'is_soma'].mean(), 1)}
    for name, mask in stages.items()])

funnel

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>What that looks like across the whole dataset</h3>

One session is a small sample, so here is the same question asked over **77 sessions from three
mice** (782149, 788406, 790322 — the mice with both a coregistration table and processed NWBs).

**Two different denominators, and they answer different questions.** An ROI-session is one
detection on one day; a neuron is a `unique_roicat_id`, which gathers all the days that cell was
detected. Counting ROI-sessions weights a cell by how many days it appeared, so it is the right
denominator for "what fraction of my traces are somas" but the wrong one for "how many cells do I
have".

Per **ROI-session** (46,956 segmented ROI-sessions in total):

| Stage | ROI-sessions | `is_soma=True` | `is_soma=False` | % soma |
| --- | --- | --- | --- | --- |
| all segmented | 46,956 | 43,458 | 3,498 | 92.6 |
| in coreg table | 36,281 | 35,739 | 542 | 98.5 |
| `matched == True` | 35,085 | 34,583 | 502 | 98.6 |
| has `cz_stack_id` | 29,394 | 29,088 | 306 | 99.0 |
| has `hcr_id` | 21,506 | 21,311 | 195 | 99.1 |

Read the other way: 82.2% of soma ROI-sessions reach the coreg table and 49.0% get an `hcr_id`,
versus 15.5% and 5.6% of non-soma ROI-sessions. **Coregistration selects for somas without being
asked to** — about ninefold. That follows from the method: the structural-stack match is a
spatial-overlap test against segmented cell bodies, so a dendrite has little to overlap with.
`is_soma` was *not* used as a coregistration criterion; the enrichment is a by-product.

<h4>The same question per neuron</h4>

Because `is_soma` is assigned per session, a single cell can be called a soma on some days and not
others. Classifying each of the 4,193 neurons in these sessions by how consistent its label is:

| Population | Neurons | always soma | mixed | never soma |
| --- | --- | --- | --- | --- |
| in coreg table | 4,193 | 3,737 (89.1%) | 396 (9.4%) | 60 (1.4%) |
| with `cz_stack_id` | 2,109 | 1,862 (88.3%) | 239 (11.3%) | 8 (0.4%) |
| with `hcr_id` | 1,525 | 1,361 (89.2%) | 159 (10.4%) | 5 (0.3%) |

So at ROI-session level only 0.9% of HCR-matched detections are non-soma — the filter looks nearly
free. At neuron level **10.4% of HCR-matched cells are mixed**, which is the number that actually
bites: filtering per session drops those cells from some sessions and keeps them in others,
silently shrinking any cross-session matched set and making the set depend on which sessions you
chose. For multi-session work, decide `is_soma` **once per cell** — majority vote over its
sessions — and apply that decision uniformly.

Two takeaways for this notebook. Filtering to somas after coregistration is cheap and worth doing,
because the ROIs it removes are dendrites that were handed a cell body's transcriptome. But do not
read the per-session number as meaning the filter is inconsequential; per cell it is not.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Subsetting to the cells we will analyze</h3>

Now, and only now, do we reduce the data. Two filters, both applied by **selecting ROI ids**:

1. `is_soma` — keep cell bodies, drop dendrites and artefacts
2. coregistered — keep ROIs with an `hcr_id`, since a cell type is the point of the notebook

`subset_session` takes the ROI table, filters it, and then uses the surviving rows' `column`
values to slice the activity array **once**. After it returns, `session['dff']` and
`session['rois']` are the same length and in the same order, and `column` is reset to index the
new array. Every plot downstream can be written without thinking about ROI identity again.

The masks and projections are read per plane on demand, so those are subset at plot time using
`roi_index` — which is why we kept it.

</div>

In [ ]:
def subset_session(session, soma_only=True, coregistered=True, require_subclass=False):
    """Restrict a session to a subset of its ROIs, keeping dff and rois consistent.

    Selects rows of the ROI table, then slices the activity matrix to match and
    renumbers `column` so it indexes the new array.
    """
    rois = session['rois']
    keep = pd.Series(True, index=rois.index)

    if soma_only:
        keep &= rois['is_soma']
    if coregistered:
        keep &= rois['hcr_id'].notna()
    if require_subclass:
        keep &= rois['subclass'].notna()

    kept = rois[keep].copy()

    subset = dict(session)
    subset['dff'] = session['dff'][:, kept['column'].values]
    if 'aligned' in session:
        subset['aligned'] = session['aligned'][:, kept['column'].values]
    if 'per_trial' in session:
        subset['per_trial'] = session['per_trial'][:, kept['column'].values]

    kept['column'] = np.arange(len(kept))          # index into the NEW dff
    subset['rois'] = kept.reset_index(drop=True)
    subset['n_segmented'] = len(rois)

    # neurons sorted by subclass, for plots that group by cell type
    typed = subset['rois'].dropna(subset=['subclass']).copy()
    typed['subclass'] = pd.Categorical(typed['subclass'], subclass_order, ordered=True)
    subset['typed'] = typed.sort_values(['subclass', 'cluster_name'])

    return subset


gratings = subset_session(gratings)

print(f"{gratings['n_segmented']} segmented ROIs -> {len(gratings['rois'])} kept "
      f"(soma AND coregistered to HCR)")
print('dff is now:', gratings['dff'].shape)
print(f"{len(gratings['typed'])} of those carry an inhibitory subclass label\n")
gratings['rois']['cell_class'].value_counts(dropna=False)

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

Read those numbers in order, because each drop is a different loss:

1. **Segmented -> soma + coregistered.** Most recorded ROIs never get an HCR match.
2. **Coregistered -> classified.** Of those that do, some come back `unassigned` — the
   transcriptomic call itself failed.
3. **Classified -> inhibitory subclass.** The gene panel is built to resolve *inhibitory*
   types. Excitatory cells are coregistered and classified but have no subclass here.

So the population in every plot below is a **thrice-filtered** subset. That is not a flaw to hide;
it is the sampling structure you have to reason about when interpreting any result. The
`n_segmented` count is kept on the session so plots can always report the denominator.

</div>

In [ ]:
gratings['typed'].groupby(['depth', 'subclass'], observed=True).size().unstack(fill_value=0)

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

Notice which planes have typed neurons and which do not.

The HCR method works on thick tissue but only penetrates the upper layers, so the deepest
planes may have few or no coregistered cells. **Typed neurons are not a random sample of
recorded neurons** — they are biased toward particular depths, and toward cells that were
tracked well across sessions.

Any comparison between typed and untyped neurons has to account for that, for example by
matching on depth.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>The expression behind the labels</h3>

Because the labels came from an AnnData, the expression that produced them is right there. It is
worth one look: the marker genes should separate the subclasses, and if they do not, the labels
are not to be trusted.

We pull the coregistered cells out of the AnnData by `hcr_id` and average the normalized
expression of each subclass's canonical marker.

</div>

In [ ]:
markers = ['Pvalb', 'Sst', 'Vip', 'Lamp5', 'Gad2', 'Slc17a7']

# var_names are probe names (R5-514-Pvalb); the plain symbol is in var['gene']
probe_of = {gene: name for name, gene in zip(adata.var_names, adata.var['gene'])}

coreg_ids = gratings['rois']['hcr_id'].dropna().astype(np.int64).astype(str)
recorded = adata[adata.obs_names.isin(coreg_ids)]

normalized = pd.DataFrame(
    recorded[:, [probe_of[g] for g in markers]].layers['normalized'],
    index=recorded.obs_names, columns=markers)
normalized['subclass'] = recorded.obs['subclass'].astype(str).values

(normalized.groupby('subclass')[markers].mean()
 .reindex(subclass_order + ['none']).round(2))

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

Read this table **down the columns**: each subclass has the highest value in its own marker gene,
which is what makes the labels credible. `Gad2` (pan-inhibitory) is high in all four, and
`Slc17a7` (excitatory) is near zero everywhere — so no excitatory cells have leaked in.

The `none` row is the coregistered cells with no subclass call. Its `Slc17a7` is also low and its
`Gad2` is substantial, so these are mostly inhibitory cells the clustering could not confidently
place — not excitatory cells. That is consistent with the `cell_class` counts above, where the
unlabelled coregistered cells were `unassigned` rather than `excitatory`. Coregistration is
biased toward the sparse inhibitory population because those are the cells the panel labels.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h2>Part 4: The four plots</h2>

Each plot is a function taking a loaded session, so we can run the same set on any session type.

<h3>Plot 1: max projections by depth, ROIs coloured by cell type</h3>

The `max_projection` is the brightest value each pixel reached over the session, so active
neurons stand out. ROI masks are stored as one image per neuron, of shape
**(neurons, height, width)**.

Typed neurons are filled in their subclass colour; the rest are left as grey outlines. Keeping
them visible matters — it shows the typed cells as a *fraction* of what was recorded.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Plot 1: max projections by depth, ROIs coloured by cell type</h3>

The `max_projection` is the brightest value each pixel reached over the session, so active
neurons stand out. ROI masks are stored as one image per ROI, of shape
**(ROIs, height, width)** — and critically, in the **full segmented order**, not the subset order,
because they are read straight from the NWB.

This is the one place we still need `roi_index`: it says which row of the mask array each kept ROI
came from. So we index the masks by `roi_index` and colour by the label carried on the same row.
The ROIs we filtered out are drawn as grey outlines, which is what shows the analyzed cells as a
*fraction* of what was segmented.

</div>

In [ ]:
def plot_max_projections(session):
    fig, axes = plt.subplots(2, 4, figsize=(16, 9.5))

    # roi_index -> subclass, for the ROIs that survived subsetting
    label_of = session['typed'].set_index(['plane', 'roi_index'])['subclass']

    for ax, plane in zip(axes.flat, session['planes_by_depth']):
        image = plane_summary_image(session, plane)
        masks = plane_masks(session, plane)          # (all segmented ROIs, h, w)

        lo, hi = np.percentile(image, [1, 99.5])
        ax.imshow(image, cmap='gray', vmin=lo, vmax=hi)

        n_typed = 0
        for i in range(masks.shape[0]):
            if (plane, i) in label_of.index:
                colour = subclass_colors[label_of[(plane, i)]]
                ax.contourf(masks[i].astype(float), levels=[0.5, 1.5], colors=[colour], alpha=0.7)
                n_typed += 1
            else:
                ax.contour(masks[i], levels=[0.5], colors='lightgray', linewidths=0.5)

        ax.set_title(f'{plane}, {session["depths"][plane]} ' + r'$\mu$m'
                     + f'\n{n_typed} of {masks.shape[0]} typed', fontsize=14)
        ax.axis('off')

    handles = [plt.Line2D([], [], marker='s', linestyle='', markersize=12,
                          color=subclass_colors[n], label=n) for n in subclass_order]
    fig.legend(handles=handles, loc='lower center', ncol=4, frameon=False)

    fig.suptitle(f"{session['date']}  --  {session['session_type']}", y=0.99)
    plt.tight_layout(rect=[0, 0.04, 1, 0.97], h_pad=3)
    plt.show()


plot_max_projections(gratings)

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Plot 2: dF/F heatmaps</h3>

One trace per neuron gets unreadable past a handful of neurons, so use a **heatmap**: each row is
one neuron, x is time, colour is &Delta;F/F.

`dff` is (frames, neurons) and we want (neurons, frames), hence the `.T`. Two panels: every
recorded neuron, then only the coregistered ones grouped by subclass. The second is a subset of
the first, which is the point — it shows how much of the population carries a cell-type label.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>A helper for the subclass groupings</h3>

Three of the four plots show neurons grouped by subclass, and in each one we need to mark where
the groups start and end. Write that once.

The groups are marked two ways: a **colour strip** down the left edge, and a divider line between
adjacent blocks. The strip is drawn in its own narrow axes so it does not eat into the heatmap.

</div>

In [ ]:
def subclass_blocks(typed):
    """Sizes, edges and centres of each subclass block in a sorted `typed` table."""
    sizes = typed['subclass'].value_counts()[subclass_order]
    edges = np.cumsum(sizes.values)

    return sizes.values, edges, edges - sizes.values / 2


def add_subclass_bar(ax, typed, label=True):
    """Draw a subclass colour strip just left of a heatmap whose y-axis is neurons."""
    sizes, edges, centres = subclass_blocks(typed)

    # a narrow axes glued to the left edge of ax. We set its limits to match rather than
    # using sharey, because sharing would make clearing ax's ticks clear the bar's too.
    bar = ax.inset_axes([-0.045, 0, 0.03, 1])
    bar.set_ylim(len(typed), 0)
    bar.set_xlim(0, 1)
    bar.set_xticks([])

    for name, start, size in zip(subclass_order, edges - sizes, sizes):
        bar.axhspan(start, start + size, color=subclass_colors[name], linewidth=0)

    if label:
        bar.set_yticks(centres)
        bar.set_yticklabels(subclass_order)
        for tick, name in zip(bar.get_yticklabels(), subclass_order):
            tick.set_color(subclass_colors[name])
            tick.set_fontweight('bold')
        bar.tick_params(length=0, pad=4)
    else:
        bar.set_yticks([])

    for side in bar.spines.values():
        side.set_visible(False)

    # dividers between adjacent blocks, drawn on the heatmap itself
    for edge in edges[:-1]:
        ax.axhline(edge, color='white', linewidth=2)

    ax.set_yticks([])

    return bar

In [ ]:
def plot_dff_heatmaps(session):
    typed = session['typed']
    time = session['timestamps'][planes[0]]

    fig, axes = plt.subplots(2, 1, figsize=(14, 10))

    axes[0].imshow(session['dff'].T, aspect='auto', cmap='magma', vmin=0, vmax=2,
                   extent=[time[0], time[-1], session['dff'].shape[1], 0])
    axes[0].set_ylabel('Neuron')
    axes[0].set_title(f"{session['dff'].shape[1]} soma ROIs coregistered to an HCR cell "
                      f"(of {session['n_segmented']} segmented)")

    axes[1].imshow(session['dff'][:, typed['column']].T, aspect='auto', cmap='magma',
                   vmin=0, vmax=2, extent=[time[0], time[-1], len(typed), 0])

    add_subclass_bar(axes[1], typed)

    axes[1].set_xlabel('Time (s)')
    axes[1].set_title(f'{len(typed)} with an inhibitory subclass label')

    fig.suptitle(f"{session['date']}  --  {session['session_type']}")
    plt.tight_layout(rect=[0.055, 0, 1, 1])
    plt.show()


plot_dff_heatmaps(gratings)

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Building a change-aligned response</h3>

Both remaining plots need the same thing: activity cut out around each image change.

The steps are: find the frame closest to each time we want, pull out those frames, subtract each
trial's own baseline so every trial starts at zero, then average.

One subtlety in the first step. `np.searchsorted` gives the first frame at or *after* the time you
ask for, so on average every sample comes from half a frame later than intended. At ~10 Hz that is
about 50 ms of systematic shift, which is enough to make a response look like it begins slightly
*before* the change. Checking the frame on either side and taking the closer one removes the bias,
which is what `nearest_frame` below does.

</div>

In [ ]:
def change_times(session):
    """Change times that fall safely inside the imaging window of every plane."""
    times = session['trials']['change_time'].dropna().values

    starts = [session['timestamps'][p][0] for p in planes]
    stops = [session['timestamps'][p][-1] for p in planes]

    return times[(times > max(starts) + 2) & (times < min(stops) - 4)]


def nearest_frame(timestamps, wanted):
    """Index of the frame closest in time to each wanted time.

    np.searchsorted alone returns the first frame at or AFTER the wanted time, which
    biases every sample half a frame late -- enough to make responses look like they
    start before the change. Comparing against the frame before it removes the bias.
    """
    after = np.clip(np.searchsorted(timestamps, wanted), 1, len(timestamps) - 1)
    before = after - 1

    closer_after = np.abs(timestamps[after] - wanted) < np.abs(timestamps[before] - wanted)

    return np.where(closer_after, after, before)


def align_to_changes(session, per_trial=False):
    """Response of every neuron around each change.

    Returns (window, neurons) averaged over changes, or (changes, neurons) averaged
    over the first second after the change when per_trial is True.
    """
    events = change_times(session)
    n_rows = len(events) if per_trial else len(window)
    result = np.zeros((n_rows, session['dff'].shape[1]))

    for plane in planes:
        cols = session['rois'].query('plane == @plane')['column'].values

        frames = nearest_frame(session['timestamps'][plane],
                               events[:, np.newaxis] + window[np.newaxis, :])

        cut = session['dff'][:, cols][frames]                       # (changes, window, neurons)
        cut = cut - cut[:, window < 0, :].mean(axis=1, keepdims=True)

        if per_trial:
            result[:, cols] = cut[:, (window >= 0) & (window <= 1), :].mean(axis=1)
        else:
            result[:, cols] = cut.mean(axis=0)

    return result

In [ ]:
gratings['aligned'] = align_to_changes(gratings)
gratings['per_trial'] = align_to_changes(gratings, per_trial=True)

print(len(change_times(gratings)), 'changes')
print('aligned  (window, neurons):', gratings['aligned'].shape)
print('per_trial (changes, neurons):', gratings['per_trial'].shape)

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Plot 3: tuning heatmap</h3>

What "tuning" means depends on the stimulus. In gratings sessions the stimulus has an
**orientation**; in natural image sessions it has an **image identity**. Both come from the trials
table, so one function handles either by reading whichever column exists.

To compare neurons we **z-score** each one across the conditions: subtract that neuron's mean and
divide by its standard deviation, so every row shows *relative* preference on the same scale.

</div>

In [ ]:
def stimulus_labels(session):
    """The stimulus shown at each change: orientation for gratings, image name for images."""
    # change_orientation is filled in on every trial including aborted ones, so we cannot
    # dropna() on it -- select the same rows we kept when aligning
    events = change_times(session)
    keep = session['trials']['change_time'].isin(events).values

    # prefer orientation: in gratings sessions BOTH columns are filled, but
    # change_image_name holds strings ('gratings_0', 'gratings_180', ...) which sort
    # alphabetically into the wrong order. change_orientation is numeric.
    for column in ['change_orientation', 'change_image_name']:
        if column not in session['trials']:
            continue
        values = session['trials'][column].values[keep]

        if len(pd.unique(values[pd.notna(values)])) > 1:
            return column, values

    raise ValueError('no varying stimulus column found')

In [ ]:
def plot_tuning(session):
    column, labels = stimulus_labels(session)
    conditions = sorted(pd.unique(labels))     # numeric for orientation, alphabetical for images

    # mean response per condition, then z-scored within each neuron
    tuning = np.stack([session['per_trial'][labels == c].mean(axis=0) for c in conditions])
    z = (tuning - tuning.mean(axis=0)) / (tuning.std(axis=0) + 1e-9)

    typed = session['typed']

    fig, axes = plt.subplots(1, 2, figsize=(13, 8.5),
                             gridspec_kw={'wspace': 0.45, 'right': 0.86})

    # all analyzed neurons, sorted by which condition they prefer
    order = np.argsort(z.argmax(axis=0))
    axes[0].imshow(z[:, order].T, aspect='auto', cmap='RdBu_r', vmin=-1.75, vmax=1.75)
    axes[0].set_ylabel('Neuron (sorted by preference)')
    axes[0].set_title(f'All {z.shape[1]} analyzed neurons')

    im = axes[1].imshow(z[:, typed['column']].T, aspect='auto', cmap='RdBu_r',
                        vmin=-1.75, vmax=1.75)

    add_subclass_bar(axes[1], typed)
    axes[1].set_title('Sorted by cell type')

    for ax in axes:
        ax.set_xticks(range(len(conditions)))
        ax.set_xticklabels([str(c).replace('.0', '') for c in conditions], rotation=45)
        ax.set_xlabel('orientation (deg)' if 'orientation' in column else 'image')

    cax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
    fig.colorbar(im, cax=cax, label='z-scored response')
    fig.suptitle(f"{session['date']}  --  {session['session_type']}")
    plt.show()


plot_tuning(gratings)

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

Look at the columns of the left panel. The **0&deg; and 180&deg;** columns resemble each other,
and so do 90&deg; and 270&deg;.

That is expected: a static grating at 0&deg; and one at 180&deg; are the same image. There are
really only two orientations here, each measured twice — which is a free reliability check, since
a genuinely tuned neuron should give the same answer both times.

Always look at the raw tuning matrix before computing a selectivity score.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Plot 4: change-aligned response by cell type</h3>

Two panels: a heatmap of every neuron's change-aligned trace, sorted by cell type, and the mean
trace per subclass with a shaded standard error.

The shading marks the stimulus. Natural image sessions flash the image for 250 ms every 0.75 s,
so we shade the **changed** image blue and the **repeated** flashes grey. Gratings in
`TRAINING_1` are static, held on screen for a couple of seconds, so there is a single blue span
and no repeats.

</div>

In [ ]:
def stimulus_spans(session):
    """Where to shade: a list of (start, stop, is_change) in seconds relative to the change."""
    stim = session['stimulus']

    duration = np.median(stim['stop_time'] - stim['start_time'])
    interval = np.median(np.diff(stim['start_time']))

    # is there another presentation inside the window we plot? if not, only the change shows
    if interval > window[-1] - window[0]:
        return [(0, duration, True)]

    # flashed: a regular train of flashes, one of which is the change at time 0
    onsets = np.arange(window[0], window[-1], interval)
    onsets = onsets - onsets[np.argmin(abs(onsets))]

    return [(o, o + duration, abs(o) < 0.01) for o in onsets]


def mark_stimulus(ax, session):
    for start, stop, is_change in stimulus_spans(session):
        ax.axvspan(start, stop, color='#4C8FCC' if is_change else '#D9D9D9',
                   alpha=0.30, linewidth=0)

    ax.axvline(0, color='#2C5F8A', linestyle='--', linewidth=1)

In [ ]:
def plot_change_response(session):
    typed = session['typed']
    aligned = session['aligned']

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    im = axes[0].imshow(aligned[:, typed['column']].T, aspect='auto', cmap='RdBu_r',
                        vmin=-0.15, vmax=0.15,
                        extent=[window[0], window[-1], len(typed), 0])

    add_subclass_bar(axes[0], typed)

    axes[0].axvline(0, color='black', linestyle='--', linewidth=1)
    axes[0].set_xlabel('Time from change (s)')
    axes[0].set_title(f'{len(typed)} typed neurons')
    fig.colorbar(im, ax=axes[0], label=r'$\Delta$F/F', fraction=0.04)

    mark_stimulus(axes[1], session)

    for name in subclass_order:
        cols = typed.query('subclass == @name')['column'].values
        if len(cols) == 0:
            continue
        trace = aligned[:, cols].mean(axis=1)
        error = aligned[:, cols].std(axis=1) / np.sqrt(len(cols))

        axes[1].plot(window, trace, color=subclass_colors[name], linewidth=2,
                     label=f'{name} (n={len(cols)})')
        axes[1].fill_between(window, trace - error, trace + error,
                             color=subclass_colors[name], alpha=0.2)

    axes[1].axhline(0, color='gray', linewidth=0.5)
    axes[1].set_xlabel('Time from change (s)')
    axes[1].set_ylabel(r'$\Delta$F/F change from baseline')
    axes[1].set_title('Mean by subclass')
    axes[1].legend(frameon=False)

    fig.suptitle(f"{session['date']}  --  {session['session_type']}")
    plt.tight_layout(rect=[0.05, 0, 1, 1])
    plt.show()


plot_change_response(gratings)

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

For the gratings session the response is a single bump that decays while the grating is
still on screen. Read the subclass means with the sample sizes in mind — some subclasses have
only a dozen neurons in one session.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h2>Part 5: A familiar natural image session</h2>

Now run the identical four plots on `OPHYS_1_images_A`, where the mouse performs the same task
with natural images it has seen for days.

What differs from gratings:

| | Gratings (`TRAINING_1`) | Natural images (`OPHYS_*`) |
| --- | --- | --- |
| What changes | orientation (4 values) | image identity (8 images) |
| Trials column | `change_orientation` | `change_image_name` |
| On screen | static, ~2.4 s, every ~10 s | flashed, 250 ms every 0.75 s |
| Blank flashes | no | yes, ~4% of flashes **omitted** |

Because the plotting functions read the stimulus from the file, none of them need changing.

</div>

In [ ]:
familiar = subset_session(add_cell_types(load_session(FAMILIAR)))
familiar['aligned'] = align_to_changes(familiar)
familiar['per_trial'] = align_to_changes(familiar, per_trial=True)

print(familiar['date'], familiar['session_type'])
print('%d of %d segmented ROIs kept, %d with a subclass label'
      % (familiar['dff'].shape[1], familiar['n_segmented'], len(familiar['typed'])))
print('stimulus:', stimulus_labels(familiar)[0])

In [ ]:
# the eight natural images of set A
pd.Series(stimulus_labels(familiar)[1]).value_counts()

In [ ]:
plot_max_projections(familiar)

In [ ]:
plot_dff_heatmaps(familiar)

In [ ]:
plot_tuning(familiar)

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Is that diagonal real?</h3>

The tuning heatmap has a striking diagonal — but be careful, because **sorting neurons by their
preferred condition produces a diagonal even in pure noise.** Each neuron's peak is put in its
own column by construction.

The honest check is **split-half reliability**: build the tuning curve twice from random halves
of the trials, and correlate the two. A neuron with a real preference gives the same answer both
times.

</div>

In [ ]:
def tuning_reliability(session, seed=1):
    """Correlate tuning curves built from two random halves of the changes."""
    column, labels = stimulus_labels(session)
    conditions = sorted(pd.unique(labels))

    order = np.random.default_rng(seed).permutation(len(labels))
    halves = [order[:len(order) // 2], order[len(order) // 2:]]

    curves = [np.stack([session['per_trial'][h][labels[h] == c].mean(axis=0) for c in conditions])
              for h in halves]

    return np.array([np.corrcoef(curves[0][:, i], curves[1][:, i])[0, 1]
                     for i in range(session['per_trial'].shape[1])])


reliability = tuning_reliability(familiar)

print('median split-half r: %.2f' % np.nanmedian(reliability))
print('fraction of neurons with r > 0.5: %.2f' % np.nanmean(reliability > 0.5))

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

The median is comfortably positive and more than half the neurons exceed 0.5, so image
preference in this session is a real, repeatable property — the diagonal is not just the sorting.

Run the same function on the gratings session to compare how reliable orientation tuning is.

</div>

In [ ]:
plot_change_response(familiar)

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

The traces **oscillate at 1.33 Hz**, which is the flash rate — the grey spans line up with each
subsequent peak, so the ringing is the stimulus, not noise. This is why we did gratings first: a
static stimulus gives one clean bump, so it is easier to recognise that the alignment is correct.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h2>Part 6: A novel image session</h2>

`OPHYS_4_images_B` is the mouse's **first exposure to a novel image set**. Same four plots.

</div>

In [ ]:
novel = subset_session(add_cell_types(load_session(NOVEL)))
novel['aligned'] = align_to_changes(novel)
novel['per_trial'] = align_to_changes(novel, per_trial=True)

print(novel['date'], novel['session_type'])
print('%d of %d segmented ROIs kept, %d with a subclass label'
      % (novel['dff'].shape[1], novel['n_segmented'], len(novel['typed'])))

# set B shares no images with set A
pd.Series(stimulus_labels(novel)[1]).value_counts()

In [ ]:
plot_max_projections(novel)

In [ ]:
plot_dff_heatmaps(novel)

In [ ]:
plot_tuning(novel)

In [ ]:
plot_change_response(novel)

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h2>Part 7: Align across sessions</h2>

The second alignment. We have the same four plots for a familiar and a novel session, but so far
each was analyzed on its own. To compare them we need to know **which neurons are the same**.

The image sets are disjoint — set A is `im061`-`im085`, set B is `im000`-`im106` — so we cannot
ask how the response to a particular image changed. We can only compare each *neuron* to itself.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Matching neurons across sessions</h3>

This is what `unique_roicat_id` is for. Recall the two scopes:

- `unique_roi_id` is **per session, per plane** — a detection on one day
- `unique_roicat_id` is **per cell, across all sessions** — the neuron itself

Note that this alignment does not involve the HCR data at all: the cross-session tracking is done
by ROICaT on the imaging data alone. The cell type comes along for the ride, because the same
coreg row carries both IDs.

Both sessions have already been subset to soma + coregistered ROIs, so every row has a
`unique_roicat_id`. Take the neurons present in both by intersecting the two sets.

One caveat carried over from the `is_soma` analysis: because we applied the soma filter **per
session**, a cell classified as a soma on one day but not the other is dropped from this
intersection. For this pair of sessions that is a small effect, but if you extend the intersection
across many sessions it compounds — that is when deciding `is_soma` once per cell starts to matter.

</div>

In [ ]:
in_familiar = set(familiar['rois']['unique_roicat_id'].dropna())
in_novel = set(novel['rois']['unique_roicat_id'].dropna())

matched_ids = sorted(in_familiar & in_novel)

print(len(in_familiar), 'coregistered in familiar')
print(len(in_novel), 'coregistered in novel')
print(len(matched_ids), 'in both')

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

Now build the lookup: for each matched neuron, which **column** does it occupy in each
session?

Setting `unique_roicat_id` as the index and then selecting `matched_ids` guarantees both tables
come out in the *same order*, so row *i* of one is the same neuron as row *i* of the other.
Getting this wrong is the easiest way to produce a meaningless result.

</div>

In [ ]:
familiar_rows = familiar['rois'].set_index('unique_roicat_id').loc[matched_ids]
novel_rows = novel['rois'].set_index('unique_roicat_id').loc[matched_ids]

pairs = pd.DataFrame({
    'unique_roicat_id': matched_ids,
    'familiar_column': familiar_rows['column'].values,
    'novel_column': novel_rows['column'].values,
    'familiar_plane': familiar_rows['plane'].values,
    'novel_plane': novel_rows['plane'].values,
    'hcr_id': familiar_rows['hcr_id'].values,
    'subclass': familiar_rows['subclass'].values,
})

# the same physical neuron must map to the same HCR cell in both sessions -- check, don't assume
consistent = (familiar_rows['hcr_id'].values == novel_rows['hcr_id'].values)
print(f'{consistent.sum()} of {len(pairs)} matched neurons agree on hcr_id across sessions')

pairs.head()

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

The `hcr_id` agreement is a genuine consistency check on the coregistration, and it should be
100%: `unique_roicat_id` and `hcr_id` are both properties of the *cell*, not of the session, so
two rows describing the same neuron must carry the same HCR cell. Any disagreement means the
coregistration is inconsistent between those two sessions, and those neurons should be dropped.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Two checks worth running</h3>

First: a matched neuron should be in the **same imaging plane** in both sessions. The
mesoscope targets the same depths each day, so a neuron jumping planes would mean the match
is wrong.

</div>

In [ ]:
same_plane = (pairs['familiar_plane'] == pairs['novel_plane'])
same_column = (pairs['familiar_column'] == pairs['novel_column'])

print(f'{same_plane.sum()} of {len(pairs)} matched neurons are in the same plane')
print(f'{same_column.sum()} of {len(pairs)} matched neurons have the same column index')

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

Almost none of them share a column index.

This is the whole reason `unique_roicat_id` exists. Segmentation runs independently on each
session, so it finds a slightly different set of ROIs in a different order every day. Column
17 in the familiar session is **not** the same neuron as column 17 in the novel session.

If you ever compare two sessions by row position, this is the number that tells you the
result is meaningless.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Does the same neuron respond consistently?</h3>

Before asking how novelty changed anything, check that a neuron's response is a stable property
of that neuron at all. We already have one number per neuron per session in `per_trial` — the
mean response in the first second after each change — so average over changes and correlate
across the matched pairs.

</div>

In [ ]:
familiar_response = familiar['per_trial'].mean(axis=0)[pairs['familiar_column']]
novel_response = novel['per_trial'].mean(axis=0)[pairs['novel_column']]

r = np.corrcoef(familiar_response, novel_response)[0, 1]

# what would we get if the pairing were wrong? shuffle one side.
shuffled = np.random.default_rng(0).permutation(novel_response)
r_shuffled = np.corrcoef(familiar_response, shuffled)[0, 1]

print(f'correctly paired: r = {r:.2f}')
print(f'shuffled pairing: r = {r_shuffled:.2f}')

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6.5))

untyped = pairs['subclass'].isna().values
ax.scatter(familiar_response[untyped], novel_response[untyped],
           color='lightgray', s=18, label=f'no cell type (n={untyped.sum()})')

for name in subclass_order:
    is_type = (pairs['subclass'] == name).values
    ax.scatter(familiar_response[is_type], novel_response[is_type],
               color=subclass_colors[name], s=45, label=f'{name} (n={is_type.sum()})')

limits = [-0.05, 0.15]
ax.plot(limits, limits, color='gray', linestyle='--', linewidth=1)

ax.set_xlim(limits)
ax.set_ylim(limits)
ax.set_xlabel('Familiar images (OPHYS_1)')
ax.set_ylabel('Novel images (OPHYS_4)')
ax.set_title(f'Change response of {len(pairs)} matched neurons\nr = {r:.2f}')
ax.legend(frameon=False, fontsize=11)
plt.show()

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

The correlation is high and the shuffled control is near zero, which tells us two things:

1. The matching is real. Random pairings give nothing.
2. How strongly a neuron responds to a change is a **stable property of that neuron**, holding
   up across a three-day gap and an entirely different image set.

Points above the diagonal responded more to novel images; below, more to familiar.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Familiar versus novel, by cell type</h3>

Now the two sessions on the same axes, using **only the matched neurons**, so any difference
cannot be caused by a different set of cells. Solid is familiar, dashed is novel.

</div>

In [ ]:
fig, axes = plt.subplots(1, len(subclass_order), figsize=(17, 4.5), sharey=True)

for ax, name in zip(axes, subclass_order):
    is_type = (pairs['subclass'] == name).values

    ax.plot(window, familiar['aligned'][:, pairs['familiar_column'][is_type]].mean(axis=1),
            color=subclass_colors[name], linewidth=2, label='familiar (A)')
    ax.plot(window, novel['aligned'][:, pairs['novel_column'][is_type]].mean(axis=1),
            color=subclass_colors[name], linestyle='--', linewidth=2, label='novel (B)')

    mark_stimulus(ax, familiar)
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.set_title(f'{name} (n={is_type.sum()})')
    ax.set_xlabel('Time from change (s)')
    ax.legend(frameon=False, fontsize=11)

axes[0].set_ylabel(r'$\Delta$F/F change')
plt.tight_layout()
plt.show()

In [ ]:
summary = pd.DataFrame({'familiar': familiar_response,
                        'novel': novel_response,
                        'subclass': pairs['subclass'].values})

summary.groupby('subclass', observed=True)[['familiar', 'novel']].agg(['count', 'mean']).round(4)

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

Read this table with the sample sizes in mind. Some subclasses have only a dozen matched
neurons in one pair of sessions, which is not enough to conclude anything about that cell type.
The way to make such a comparison convincing is to repeat it over many session pairs and many
mice, which is what the full dataset supports.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>One neuron, two sessions, one transcriptome</h3>

The plot that shows what all this ID-matching bought us: the same physical neuron, recorded on
two days three days apart under two different image sets, with its cell type and its measured
gene expression — three separate measurements resolved to one cell.

</div>

In [ ]:
has_type = pairs['subclass'].notna().values
pick = np.where(has_type)[0][np.argmax(familiar_response[has_type])]

cell = pairs.iloc[pick]

print(cell['unique_roicat_id'], '|', cell['subclass'], '| hcr_id', int(cell['hcr_id']))
print('familiar: plane', cell['familiar_plane'], 'column', cell['familiar_column'])
print('novel:    plane', cell['novel_plane'], 'column', cell['novel_column'])

# its top-expressed genes, straight out of the AnnData
one_cell = adata[str(int(cell['hcr_id']))]
expression = pd.Series(np.asarray(one_cell.layers['normalized']).ravel(),
                       index=adata.var['gene'].values)
print('\ntop genes:')
print(expression.sort_values(ascending=False).head(6).round(2).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

for session, column, name, style in [
        (familiar, cell['familiar_column'], 'familiar (A)', '-'),
        (novel, cell['novel_column'], 'novel (B)', '--')]:

    plane = session['rois']['plane'][column]
    axes[0].plot(session['timestamps'][plane], session['dff'][:, column],
                 linewidth=0.4, label=f"{name} -- {session['date']}")

    axes[1].plot(window, session['aligned'][:, column], color='black',
                 linestyle=style, linewidth=2, label=name)

axes[0].set_xlabel('Time from session start (s)')
axes[0].set_ylabel(r'$\Delta$F/F')
axes[0].set_title('Whole session')
axes[0].legend(frameon=False, fontsize=11)

mark_stimulus(axes[1], familiar)
axes[1].set_xlabel('Time from change (s)')
axes[1].set_title('Change-aligned')
axes[1].legend(frameon=False)

fig.suptitle(f"One {cell['subclass']} neuron (hcr_id {int(cell['hcr_id'])}), two sessions")
plt.tight_layout()
plt.show()

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h2>Where to go next</h2>

The pieces are now in place to ask the questions this dataset was built for.

- **Follow neurons through the whole curriculum.** `load_session` plus the
  `unique_roicat_id` intersection generalizes to any number of sessions. Intersect across all
  of them to get the neurons tracked from naive to expert, and watch their responses change.
  Decide `is_soma` once per cell rather than per session before you do.
- **The novelty effect properly.** Compare `OPHYS_1_images_A` (familiar), `OPHYS_4_images_B`
  (novel), and `OPHYS_6_images_B` (set B once it has become familiar). The three-way comparison
  separates novelty from image set.
- **Change what you keep.** `subset_session(session, soma_only=False)` or
  `coregistered=False` re-runs any plot on a different population — for instance, to check that
  a result on coregistered cells also holds in the full recorded population.
- **Use expression as a continuous variable.** Everything above treated the cell type as a
  discrete label, but `adata.layers['normalized']` gives graded expression per cell. Correlating a
  functional metric against a single gene avoids committing to a clustering at all.
- **Use the finer clusters.** `adata.obs['cluster']` has ~30 clusters where `subclass` has four.
  There are fewer coregistered neurons per cluster, so this needs pooling across mice.
- **Omitted flashes.** Align to `omitted == True` in the stimulus table instead of to changes.
- **Hits versus misses.** Split `change_time` by trial outcome. Interpret differences beyond
  about a second after the change with care: by then the mouse has licked and consumed reward,
  so movement and reward are mixed in with vision.
- **Other mice.** `sessions['subject_id'].unique()` lists them. Each has its own coregistration
  table under the same asset and its own HCR AnnData.

Three cautions carried over from earlier parts. First, coregistered neurons are not a random
sample of segmented ROIs — they are biased toward superficial depths, toward somas, and toward
cells that tracked reliably, so compare like with like. Second, all joins go through
`unique_roi_id` and `unique_roicat_id`; the moment you find yourself indexing one session's array
with another session's positions, stop. Third, a selectivity index built as a ratio saturates when
one of the two responses is negative; check the sign before trusting the value.

</div>